## Install Dependencies

In [ ]:
# Install dependencies
%pip install -q amplpy matplotlib

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["gurobi", "xpress", "highs", "gokestrel"], # modules to install
    license_uuid="default") # license to use

### AMPL Formulation
##### Model Fragment (Flat Sets)

In [18]:
%%ampl_eval     # Run AMPL code cell
reset;          # Clear model, data, and options

# ===== LITERAL SET =====
set SHIFTS = {"Day", "Evening", "Night"};  # Fixed set

# ===== ARITHMETIC RANGE SET =====
param start integer;                        # Lower bound
param finish > start integer;               # Upper bound
param interval > 0 integer;                 # Step size

set YEARS = start .. finish by interval;    # Computed range set

# ===== FILTERED COMPREHENSION SET =====
set PROD;                                   # Product set
param cost {PROD} >= 0;                     # Cost per product

set CHEAP = {p in PROD: cost[p] < 10};      # Products with cost < 10

# ===== SET ALGEBRA COMBINATIONS =====
set PROD_A;                                 # First product set
set PROD_B;                                 # Second product set

set ALL_PROD  = PROD_A union PROD_B;        # Union
set COMMON    = PROD_A inter PROD_B;        # Intersection
set ONLY_A    = PROD_A diff PROD_B;         # Difference
set EXCLUSIVE = PROD_A symdiff PROD_B;      # Symmetric difference
set PAIRS     = PROD_A cross PROD_B;        # Cartesian product

##### Data(Flat Sets)

In [14]:
%%ampl_eval     # Run AMPL code cell
reset data;     # Clear only data, keep model declarations
data;           # Start data section

# ===== RANGE DATA =====
param start := 1;
param finish := 10;
param interval := 1;

# ===== PRODUCT DATA =====
set PROD := coils plate rods;

param cost := coils 5, plate 15, rods  25;

# ===== SET ALGEBRA DATA =====
set PROD_A := bands coils;
set PROD_B := coils plate rods;

# ===== DISPLAY RESULTS =====
display SHIFTS, YEARS, CHEAP,
ALL_PROD, COMMON, ONLY_A, EXCLUSIVE, PAIRS;

set SHIFTS := Day Evening Night;

set YEARS := 1 2 3 4 5 6 7 8 9 10;

set CHEAP := coils;

set ALL_PROD := bands coils plate rods;

set COMMON := coils;

set ONLY_A := bands;

set EXCLUSIVE := bands plate rods;

set PAIRS :=
(bands,coils)   (bands,rods)    (coils,plate)
(bands,plate)   (coils,coils)   (coils,rods);



### Indexed Sets
##### Model Fragment (Indexed Sets)

In [15]:
%%ampl_eval     # Run AMPL code cell
reset;          # Clear model, data, and options

# ===== INDEX SETS =====
set WORKERS;                                        # Workers
set CALS;                                           # Calendar cases
set PLANTS;                                         # Plants
set GROUPS;                                         # Product groups

# ===== INDEXED LITERAL SET =====
set SHIFT_TYPES = {"Day", "Evening", "Night"};      # Fixed allowed shifts
set SHIFTS {w in WORKERS} within SHIFT_TYPES;       # Shifts for each worker

# ===== INDEXED ARITHMETIC RANGE SET =====
param start {c in CALS} integer;                    # Lower bound
param finish {c in CALS} > start[c] integer;        # Upper bound
param interval {c in CALS} > 0 integer;             # Step size

set YEARS {c in CALS} = start[c] .. finish[c] by interval[c];   # Years for each case

# ===== INDEXED FILTERED COMPREHENSION SET =====
set PROD {pl in PLANTS};                            # Products at each plant
param cost {pl in PLANTS, p in PROD[pl]} >= 0;      # Product cost
set CHEAP {pl in PLANTS} = {p in PROD[pl]: cost[pl,p] < 10}; # Cheap products by plant

# ===== INDEXED SET ALGEBRA COMBINATIONS =====
set PROD_A {g in GROUPS};                           # First set by group
set PROD_B {g in GROUPS};                           # Second set by group

set ALL_PROD {g in GROUPS} = PROD_A[g] union PROD_B[g]; # Union
set COMMON {g in GROUPS} = PROD_A[g] inter PROD_B[g];   # Intersection
set ONLY_A {g in GROUPS} = PROD_A[g] diff PROD_B[g];    # Difference
set EXCLUSIVE {g in GROUPS} = PROD_A[g] symdiff PROD_B[g]; # Symmetric difference
set PAIRS {g in GROUPS} dimen 2 = PROD_A[g] cross PROD_B[g];# Cartesian product

#### Data (Indexed Sets)


In [16]:
%%ampl_eval     # Run AMPL code cell
reset data;     # Clear only data, keep model declarations
data;           # Start data section

# ===== INDEX DATA =====
set WORKERS := Alice Bob;
set CALS := Short Long;
set PLANTS := P1 P2;
set GROUPS := G1 G2;

# ===== INDEXED LITERAL SET DATA =====
set SHIFTS[Alice] := Day Evening;
set SHIFTS[Bob]   := Night;

# ===== INDEXED RANGE DATA =====
param start := Short 1, Long  2020;

param finish := Short 10, Long  2025;

param interval := Short 1, Long  1;

# ===== INDEXED PRODUCT DATA =====
set PROD[P1] := coils plate rods;
set PROD[P2] := bands coils wire;

param cost :=
    P1 coils 5
    P1 plate 15
    P1 rods  25
    P2 bands 8
    P2 coils 12
    P2 wire  6;

# ===== INDEXED SET ALGEBRA DATA =====
set PROD_A[G1] := bands coils;
set PROD_B[G1] := coils plate rods;

set PROD_A[G2] := wire cable;
set PROD_B[G2] := cable fiber switch;

# ===== DISPLAY RESULTS =====
display SHIFTS;
display YEARS;
display CHEAP;
display ALL_PROD, COMMON, ONLY_A, EXCLUSIVE, PAIRS;

set SHIFTS[Alice] := Day Evening;
set SHIFTS[Bob] := Night;

set YEARS[Short] := 1 2 3 4 5 6 7 8 9 10;
set YEARS[Long] := 2020 2021 2022 2023 2024 2025;

set CHEAP[P1] := coils;
set CHEAP[P2] := bands wire;

set ALL_PROD[G1] := bands coils plate rods;
set ALL_PROD[G2] := wire cable fiber switch;

set COMMON[G1] := coils;
set COMMON[G2] := cable;

set ONLY_A[G1] := bands;
set ONLY_A[G2] := wire;

set EXCLUSIVE[G1] := bands plate rods;
set EXCLUSIVE[G2] := wire fiber switch;

set PAIRS[G1] :=
(bands,coils)   (bands,rods)    (coils,plate)
(bands,plate)   (coils,coils)   (coils,rods);
set PAIRS[G2] :=
(wire,cable)     (wire,switch)    (cable,fiber)
(wire,fiber)     (cable,cable)    (cable,switch);



### AMPLPY Formulation
This section reuses the ***AMPL model(Flat Sets) fragment*** declared above and loads the same data from Python.
#### Data

In [23]:
# Clear only data, keep model declarations
ampl.eval("reset data;")

# ===== RANGE DATA =====
ampl.param["start"] = 1
ampl.param["finish"] = 10
ampl.param["interval"] = 1

# ===== PRODUCT DATA =====
ampl.set["PROD"] = ["coils", "plate", "rods"]

ampl.param["cost"] = {
    "coils": 5,
    "plate": 15,
    "rods": 25,
}

# ===== SET ALGEBRA DATA =====
ampl.set["PROD_A"] = ["bands", "coils"]
ampl.set["PROD_B"] = ["coils", "plate", "rods"]

# ===== DISPLAY RESULTS =====
ampl.eval("""
display SHIFTS, YEARS, CHEAP,
        ALL_PROD, COMMON, ONLY_A, EXCLUSIVE, PAIRS;
""")

set SHIFTS := Day Evening Night;

set YEARS := 1 2 3 4 5 6 7 8 9 10;

set CHEAP := coils;

set ALL_PROD := bands coils plate rods;

set COMMON := coils;

set ONLY_A := bands;

set EXCLUSIVE := bands plate rods;

set PAIRS :=
(bands,coils)   (bands,rods)    (coils,plate)
(bands,plate)   (coils,coils)   (coils,rods);

